# SQL Top-N Interview Problems

Companion to the [complete article](https://todatabeyond.com/blog/unlocking-the-power-of-sql-how-to-ace-top-n-problem-questions-in-interviews). Run all cells in order in a Python 3.11+ Jupyter/Colab kernel with SQLite 3.25+. No extra package install, keys, server, GPU or data downloads. Python's sqlite3 module executes real SQL in isolated in-memory databases. Setup registers the small %%sql cell magic used below; there is no hidden ipython-sql dependency.

The source supplies no tables/data. Setup creates clearly synthetic values and restaurant-review fixtures. Same restaurant names occur in different cities; averages group by both name and city. Two reviews per restaurant produce fractional means. North has duplicate scores above the cutoff and tied fifth-distinct-score restaurants; West has a tie crossing the fifth row; South has fewer than five restaurants. These are teaching data, not real restaurant reviews.

All six source SQL examples are maintained: replace the reserved placeholder Table with T, fix missing commas, unify avg_ratings CTE references and return avg_rating rather than nonexistent rating. Use row_num consistently. A name tie-breaker makes ROW_NUMBER deterministic; DENSE_RANK orders only by average score so ties remain peers. Final ORDER BY makes printed results reproducible without changing rank membership.

The source's 'with ties' query means the top five distinct average scores (DENSE_RANK), not RANK<=5 or SQL FETCH FIRST 5 ROWS WITH TIES. Such queries can return fewer than five rows in small groups. The original prose is preserved in the blog; differences are in the internal website review, not public editorial notes. AVG already returns a real average in SQLite; retaining *1.0 makes the original cast intent explicit. Source MySQL wording is historical; this run validates SQLite, not a MySQL server.

Expected results below are obtained from actual SQL execution, not fabricated output. Full independently computed comparisons and edge cases run at the end. Git commits keep outputs empty and execution counts null. The final cell closes the in-memory databases; rerun setup before running queries again.


In [ ]:
import sqlite3
from collections import defaultdict
from IPython.core.magic import register_cell_magic

assert sqlite3.sqlite_version_info >= (3, 25, 0), 'SQLite window functions require 3.25+'
connection = sqlite3.connect(':memory:')
connection.execute('CREATE TABLE T (value INTEGER NOT NULL UNIQUE)')
values_fixture = [7, 12, 3, 8, 21, 5, 10]
connection.executemany('INSERT INTO T VALUES (?)', [(v,) for v in values_fixture])
connection.execute('CREATE TABLE rating (id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT NOT NULL, rating INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5))')
# Synthetic teaching data: two independent reviews per restaurant, not real ratings.
restaurants = [
    ('A', 'North', [5, 5]), ('B', 'North', [4, 5]),
    ('C', 'North', [4, 5]), ('D', 'North', [4, 4]),
    ('E', 'North', [3, 4]), ('F', 'North', [3, 3]), ('G', 'North', [3, 3]),
    ('P', 'West', [5, 5]), ('Q', 'West', [4, 5]), ('R', 'West', [4, 4]),
    ('S', 'West', [3, 4]), ('T', 'West', [3, 3]), ('U', 'West', [3, 3]),
    ('V', 'West', [2, 3]), ('W', 'West', [2, 2]),
    ('A', 'South', [2, 2]), ('B', 'South', [5, 5]),
]
rating_fixture = [(i + 1, name, city, score)
                  for i, (name, city, score) in enumerate(
                      (name, city, score) for name, city, scores in restaurants for score in scores)]
connection.executemany('INSERT INTO rating VALUES (?, ?, ?, ?)', rating_fixture)
query_results = {}
query_texts = {}

def print_result(columns, rows):
    text_rows = [[str(value) if value is not None else 'NULL' for value in row] for row in rows]
    widths = [max([len(column)] + [len(row[i]) for row in text_rows]) for i, column in enumerate(columns)]
    def format_row(row):
        return ' | '.join(value.ljust(width) for value, width in zip(row, widths)).rstrip()
    print(format_row(columns))
    print('-+-'.join('-' * width for width in widths))
    for row in text_rows:
        print(format_row(row))
    print('Rows:', len(rows))

@register_cell_magic
def sql(line, cell):
    key = line.strip()
    cursor = connection.execute(cell)
    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()
    query_texts[key] = cell.strip()
    query_results[key] = (columns, rows)
    print_result(columns, rows)

print('SQLite:', sqlite3.sqlite_version)
print('In-memory synthetic fixtures: 7 distinct values; 34 reviews, 17 restaurants, 3 cities.')
print('SQL cell magic registered. No files, external database, credentials or downloads.')


## Query 1


In [ ]:
%%sql example-0
SELECT value
FROM T
ORDER BY value DESC
LIMIT 1 OFFSET 4;


## Query 2


In [ ]:
%%sql example-1
SELECT value
FROM (
    SELECT value, ROW_NUMBER() OVER (ORDER BY value DESC) AS row_num
    FROM T
) AS ranking_table
WHERE row_num = 5;


## Query 3


In [ ]:
%%sql example-2
SELECT name, city, AVG(rating * 1.0) AS average_rating
FROM rating
GROUP BY name, city
ORDER BY city, average_rating DESC, name;


## Query 4


In [ ]:
%%sql example-3
WITH avg_ratings AS (
    SELECT name, city, AVG(rating * 1.0) AS avg_rating
    FROM rating
    GROUP BY name, city
)
SELECT name, city, avg_rating,
       ROW_NUMBER() OVER (
           PARTITION BY city ORDER BY avg_rating DESC, name
       ) AS row_num
FROM avg_ratings
ORDER BY city, row_num;


## Query 5


In [ ]:
%%sql example-4
WITH avg_ratings AS (
    SELECT name, city, AVG(rating * 1.0) AS avg_rating
    FROM rating
    GROUP BY name, city
), rating_rank AS (
    SELECT name, city, avg_rating,
           ROW_NUMBER() OVER (
               PARTITION BY city ORDER BY avg_rating DESC, name
           ) AS row_num
    FROM avg_ratings
)
SELECT name, city, avg_rating
FROM rating_rank
WHERE row_num <= 5
ORDER BY city, avg_rating DESC, name;


## Query 6


In [ ]:
%%sql example-5
WITH avg_ratings AS (
    SELECT name, city, AVG(rating * 1.0) AS avg_rating
    FROM rating
    GROUP BY name, city
), rating_rank AS (
    SELECT name, city, avg_rating,
           DENSE_RANK() OVER (
               PARTITION BY city ORDER BY avg_rating DESC
           ) AS row_num
    FROM avg_ratings
)
SELECT name, city, avg_rating
FROM rating_rank
WHERE row_num <= 5
ORDER BY city, avg_rating DESC, name;


## Regression checks


In [ ]:
assert query_results['example-0'][1] == query_results['example-1'][1] == [(7,)]
grouped = defaultdict(list)
for _, name, city, score in rating_fixture:
    grouped[(name, city)].append(score)
averages = {key: sum(scores) / len(scores) for key, scores in grouped.items()}
expected_averages = [(name, city, average) for (name, city), average in
                     sorted(averages.items(), key=lambda item: (item[0][1], -item[1], item[0][0]))]
assert query_results['example-2'][1] == expected_averages
expected_ranked = []
for city in sorted({city for _, city in averages}):
    group = [row for row in expected_averages if row[1] == city]
    expected_ranked.extend((*row, i + 1) for i, row in enumerate(group))
assert query_results['example-3'][1] == expected_ranked

def expected_top_n(n, dense):
    result = []
    for city in sorted({city for _, city in averages}):
        group = [row for row in expected_averages if row[1] == city]
        scores = sorted({row[2] for row in group}, reverse=True)
        result.extend([row for row in group if scores.index(row[2]) < n] if dense else group[:n])
    return result

for key, dense in [('example-4', False), ('example-5', True)]:
    assert query_results[key][1] == expected_top_n(5, dense)
    parameterized = query_texts[key].replace('row_num <= 5', 'row_num <= :n')
    for n in range(9):
        assert connection.execute(parameterized, {'n': n}).fetchall() == expected_top_n(n, dense)
        # Reinsert the same reviews backwards into an independent in-memory DB.
        reverse_db = sqlite3.connect(':memory:')
        reverse_db.execute('CREATE TABLE rating (id INTEGER, name TEXT, city TEXT, rating INTEGER)')
        reverse_db.executemany('INSERT INTO rating VALUES (?, ?, ?, ?)', reversed(rating_fixture))
        assert reverse_db.execute(parameterized, {'n': n}).fetchall() == expected_top_n(n, dense)
        reverse_db.close()

row_top = query_results['example-4'][1]
dense_top = query_results['example-5'][1]
assert len(row_top) == 12 and len(dense_top) == 15
assert [name for name, city, _ in row_top if city == 'West'] == ['P', 'Q', 'R', 'S', 'T']
assert [name for name, city, _ in dense_top if city == 'West'] == ['P', 'Q', 'R', 'S', 'T', 'U']
assert [name for name, city, _ in dense_top if city == 'North'] == list('ABCDEFG')
assert [row for row in dense_top if row[1] == 'South'] == [('B', 'South', 5.0), ('A', 'South', 2.0)]
# RANK includes ties at the fifth row position, not the top five distinct scores.
rank_sql = query_texts['example-5'].replace('DENSE_RANK()', 'RANK()')
rank_top = connection.execute(rank_sql).fetchall()
assert len([row for row in rank_top if row[1] == 'North']) == 5
assert len([row for row in rank_top if row[1] == 'West']) == 6
assert len(rank_top) == 13
assert rank_top != dense_top

empty = sqlite3.connect(':memory:')
empty.execute('CREATE TABLE T (value INTEGER)')
empty.execute('CREATE TABLE rating (id INTEGER, name TEXT, city TEXT, rating INTEGER)')
for sql_text in query_texts.values():
    assert empty.execute(sql_text).fetchall() == []
for count in range(1, 5):
    empty.execute('INSERT INTO T VALUES (?)', (count,))
    for key in ['example-0', 'example-1']:
        assert empty.execute(query_texts[key]).fetchall() == []
empty.close()
assert connection.execute('SELECT value FROM T ORDER BY value').fetchall() == [(v,) for v in sorted(values_fixture)]
assert connection.execute('SELECT * FROM rating ORDER BY id').fetchall() == rating_fixture
connection.close()
print('All six SQL statements and complete result comparisons passed.')
print('Fifth largest value: 7; ROW_NUMBER top five: 12 rows; DENSE_RANK top five distinct scores: 15 rows.')
print('RANK cutoff: 13 rows. Tested N=0..8, ties, two-restaurant city, empty/short inputs and reverse insertion order.')
print('All work stayed in isolated in-memory databases; source fixtures remained unchanged.')
